# Training YOLO11 with Roboflow Dataset

- URL Dataset ที่ให้มาชี้ไปยังชุดข้อมูล `Extra-Fruit&Vegetable-PromaxPlus-(Merge3Projects)` จาก Roboflow มันมาจาก การเอา FoodItems + Fruits and Vegetables + fruit-and-vegetable เอามารวมกัน
- Dataset นี้มี **50 classes** 
- Notebook นี้จึงมีขั้นตอน **auto-split** เพื่อสร้าง `train / valid / test` ให้พร้อมก่อนเริ่ม train


In [ ]:
!nvidia-smi


## Workflow

1. ดาวน์โหลด dataset จาก Roboflow URL ที่ผู้ใช้ให้มา ถ้า Run บน windows จะอยู่ C:\content แทนถ้าไม่อยู่ในไฟล์โครงการ
2. ตรวจสอบโครงสร้าง dataset และ auto-split หากไม่มี `valid/test`
3. สร้าง `data.yaml` สำหรับ YOLO11
4. train โมเดลด้วย augmentation ที่กำหนด
5. ประเมินผลและสรุปค่า `mAP@50` กับ `mAP@50-95`


In [ ]:
%pip install -q ultralytics pyyaml requests tqdm


In [ ]:
import os
import random
import requests
import shutil
import sys
import zipfile
from pathlib import Path

import yaml
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from IPython.display import Markdown, display

DATASET_URL = "https://app.roboflow.com/ds/uqqNG7E1R4?key=A2kwMYodGT"
FORCE_REDOWNLOAD = False
FORCE_REEXTRACT = False


def is_colab_runtime() -> bool:
    return "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG") is not None


LEGACY_BASE_DIR = Path("/content")
DEFAULT_BASE_DIR = LEGACY_BASE_DIR if is_colab_runtime() else Path.cwd() / "runtime_data"
BASE_DIR = DEFAULT_BASE_DIR
legacy_download_path = LEGACY_BASE_DIR / "roboflow_yolo11.zip"

if (
    not is_colab_runtime()
    and DEFAULT_BASE_DIR != LEGACY_BASE_DIR
    and legacy_download_path.exists()
    and zipfile.is_zipfile(legacy_download_path)
):
    BASE_DIR = LEGACY_BASE_DIR
    print(f"Status: reusing legacy cache dir -> {BASE_DIR}")

BASE_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_PATH = BASE_DIR / "roboflow_yolo11.zip"
RAW_DIR = BASE_DIR / "roboflow_raw"
WORK_DIR = BASE_DIR / "custom_data_yolo11"
SEED = 42
TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO = 0.10


def find_data_yaml(base_dir: Path):
    direct_path = base_dir / "data.yaml"
    if direct_path.exists():
        return direct_path

    candidates = sorted(base_dir.rglob("data.yaml"))
    return candidates[0] if candidates else None


print("Dataset URL:", DATASET_URL)
print("Runtime base dir:", BASE_DIR)
print("Download path:", DOWNLOAD_PATH)
print("Cached zip exists:", DOWNLOAD_PATH.exists())
print("Extracted raw dataset exists:", find_data_yaml(RAW_DIR) is not None)
print("Prepared split dataset exists:", (WORK_DIR / "data.yaml").exists())
print(f"Flags -> FORCE_REDOWNLOAD={FORCE_REDOWNLOAD}, FORCE_REEXTRACT={FORCE_REEXTRACT}")
print("Status: ready for download/extract step")


def download_with_progress(url: str, output_path: Path, chunk_size: int = 1024 * 1024):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    print("Status: connecting to dataset server...")

    with requests.get(url, stream=True, timeout=(30, 600), allow_redirects=True) as response:
        response.raise_for_status()
        total_size = int(response.headers.get("content-length", 0))
        downloaded = 0
        last_reported_pct = -1

        if total_size > 0:
            print(f"Status: downloading dataset ({total_size / (1024 ** 3):.2f} GB)")
        else:
            print("Status: downloading dataset (size unknown)")

        with open(output_path, "wb") as file_obj, tqdm(
            total=total_size if total_size > 0 else None,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            mininterval=0.5,
            dynamic_ncols=True,
            leave=True,
            desc="Download",
        ) as progress_bar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if not chunk:
                    continue

                file_obj.write(chunk)
                size = len(chunk)
                downloaded += size
                progress_bar.update(size)

                if total_size > 0:
                    pct = int(downloaded * 100 / total_size)
                    if pct >= last_reported_pct + 10:
                        print(f"Status: {pct}% ({downloaded / (1024 ** 2):,.0f} MB / {total_size / (1024 ** 2):,.0f} MB)")
                        last_reported_pct = pct

    print(f"Status: download complete -> {output_path}")
    if not zipfile.is_zipfile(output_path):
        raise zipfile.BadZipFile(f"Downloaded file is incomplete or invalid: {output_path}")


In [ ]:
missing = [
    name for name in ("DOWNLOAD_PATH", "RAW_DIR", "download_with_progress", "find_data_yaml", "zipfile", "shutil")
    if name not in globals()
]
if missing:
    raise RuntimeError("กรุณารัน cell 5 ใหม่ก่อน cell 6; ตัวแปรที่ยังไม่มี: " + ", ".join(missing))

if FORCE_REDOWNLOAD and DOWNLOAD_PATH.exists():
    print(f"Status: removing cached archive -> {DOWNLOAD_PATH}")
    DOWNLOAD_PATH.unlink()

if FORCE_REEXTRACT and RAW_DIR.exists():
    print(f"Status: removing extracted dataset -> {RAW_DIR}")
    shutil.rmtree(RAW_DIR)

if DOWNLOAD_PATH.exists() and not zipfile.is_zipfile(DOWNLOAD_PATH):
    print(f"Status: cached archive is invalid or incomplete -> {DOWNLOAD_PATH}")
    DOWNLOAD_PATH.unlink()

if DOWNLOAD_PATH.exists():
    print(f"Status: using existing archive -> {DOWNLOAD_PATH}")
else:
    download_with_progress(DATASET_URL, DOWNLOAD_PATH)

if not zipfile.is_zipfile(DOWNLOAD_PATH):
    raise zipfile.BadZipFile(f"Archive is still invalid after download: {DOWNLOAD_PATH}")

data_yaml_path = find_data_yaml(RAW_DIR)
if data_yaml_path is not None:
    RAW_DATASET_DIR = data_yaml_path.parent
    print(f"Status: using existing extracted dataset -> {RAW_DATASET_DIR}")
else:
    print("Status: extracting zip file...")
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DOWNLOAD_PATH, "r") as zip_ref:
        zip_ref.extractall(RAW_DIR)
    print("Status: extraction completed")

    data_yaml_path = find_data_yaml(RAW_DIR)
    if data_yaml_path is None:
        raise FileNotFoundError("แตกไฟล์เสร็จแล้วแต่ยังไม่พบ data.yaml ใน raw dataset")

    RAW_DATASET_DIR = data_yaml_path.parent
    print(f"Status: detected extracted dataset -> {RAW_DATASET_DIR}")

print("Status: download/extraction step completed")


In [ ]:
missing = [name for name in ("RAW_DIR", "yaml", "find_data_yaml") if name not in globals()]
if missing:
    raise RuntimeError("กรุณารัน cell 5 ก่อน cell 7; ตัวแปรที่ยังไม่มี: " + ", ".join(missing))

data_yaml_path = find_data_yaml(RAW_DIR)
if data_yaml_path is None:
    raise FileNotFoundError("ยังไม่พบ data.yaml ใน raw dataset กรุณารัน cell 6 ให้สำเร็จก่อน")

RAW_DATASET_DIR = data_yaml_path.parent
readme_candidates = [
    RAW_DATASET_DIR / "README.roboflow.txt",
    RAW_DATASET_DIR / "README.dataset.txt",
    RAW_DIR / "README.roboflow.txt",
]

for readme_path in readme_candidates:
    if readme_path.exists():
        print(readme_path.read_text(encoding="utf-8"))
        break

raw_data = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8"))
print("\nข้อมูลจาก data.yaml")
print("Raw dataset dir:", RAW_DATASET_DIR)
print(raw_data)


## เตรียมข้อมูลและคำนวณสัดส่วน Train / Valid / Test

Cell ถัดไปจะ:
- ตรวจว่าชุดข้อมูลมี `train`, `valid`, `test` ครบหรือไม่
- ถ้าไม่ครบ จะรวมรูปทั้งหมดแล้ว split ใหม่ตามสัดส่วน `80/10/10`
- แสดงจำนวนรูปภาพและเปอร์เซ็นต์ในแต่ละ split


In [ ]:
import os
import random
import shutil
from pathlib import Path

import yaml
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from IPython.display import Markdown, display

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".gif"}
JPEG_EXTS = {".jpg", ".jpeg"}
JPG_SAVE_KWARGS = {"format": "JPEG", "quality": 95, "subsampling": 0}
SPLITS = ("train", "valid", "test")
PROGRESS_REPORT_EVERY = 500

missing = [
    name for name in ("RAW_DIR", "WORK_DIR", "SEED", "TRAIN_RATIO", "VALID_RATIO", "TEST_RATIO", "raw_data", "find_data_yaml")
    if name not in globals()
]
if missing:
    raise RuntimeError(
        "RUN required cells first; missing variables: " + ", ".join(missing)
    )

raw_dataset_dir = globals().get("RAW_DATASET_DIR")
if raw_dataset_dir is None:
    data_yaml_path = find_data_yaml(RAW_DIR)
    if data_yaml_path is None:
        raise FileNotFoundError("data.yaml not found in raw dataset; run the dataset download/extract cells first")
    raw_dataset_dir = data_yaml_path.parent
    RAW_DATASET_DIR = raw_dataset_dir


def list_images(folder: Path):
    if not folder.exists():
        return []
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS
    )


def label_for_image(image_path: Path) -> Path:
    return image_path.parent.parent / "labels" / f"{image_path.stem}.txt"


def jpg_name_for_image(image_path: Path) -> str:
    return f"{image_path.stem}.jpg"


def ensure_empty_split_dirs(base_dir: Path):
    if base_dir.exists():
        shutil.rmtree(base_dir)
    for split in SPLITS:
        (base_dir / split / "images").mkdir(parents=True, exist_ok=True)
        (base_dir / split / "labels").mkdir(parents=True, exist_ok=True)


def link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def rgb_image_for_jpg(image: Image.Image) -> Image.Image:
    needs_alpha_flatten = image.mode in {"RGBA", "LA"} or (
        image.mode == "P" and "transparency" in image.info
    )
    if needs_alpha_flatten:
        rgba_image = image.convert("RGBA")
        background = Image.new("RGB", rgba_image.size, (255, 255, 255))
        background.paste(rgba_image, mask=rgba_image.getchannel("A"))
        return background
    return image.convert("RGB")


def prepare_image_as_jpg(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        raise FileExistsError(f"JPG target already exists: {dst}")

    try:
        with Image.open(src) as image:
            source_format = image.format or "UNKNOWN"
            if source_format == "JPEG":
                link_or_copy(src, dst)
                return "copied_jpeg"

            if getattr(image, "is_animated", False):
                image.seek(0)

            rgb_image = rgb_image_for_jpg(image)
            rgb_image.save(dst, **JPG_SAVE_KWARGS)
            return "converted_to_jpg"
    except (UnidentifiedImageError, OSError) as exc:
        if src.suffix.lower() in JPEG_EXTS:
            print(f"Warning: could not fully decode {src.name} ({exc}); copying original bytes as JPG")
            link_or_copy(src, dst)
            return "copied_jpeg_unverified"
        raise RuntimeError(f"Failed to convert image to JPG: {src}") from exc


def split_counts(base_dir: Path):
    return {
        split: len(list_images(base_dir / split / "images"))
        for split in SPLITS
    }


def prepare_dataset(raw_dir: Path, work_dir: Path, raw_data: dict):
    existing_counts = split_counts(raw_dir)
    print("Status: existing split counts ->", existing_counts)
    has_complete_splits = all(existing_counts[split] > 0 for split in SPLITS)

    source_split_images = {}
    all_images = []
    for split in SPLITS:
        split_folder = raw_dir / split / "images"
        split_images = list_images(split_folder)
        source_split_images[split] = split_images
        all_images.extend(split_images)
        print(f"Status: scanned {split_folder} -> {len(split_images):,} images")

    if not all_images:
        raise FileNotFoundError("No images found in dataset")

    if has_complete_splits:
        mapping = source_split_images
        print("Status: split exists -> preserve original split and normalize images to JPG")
    else:
        print("Status: valid/test missing -> create new 80/10/10 split and normalize images to JPG")
        rng = random.Random(SEED)
        rng.shuffle(all_images)

        total = len(all_images)
        train_end = int(total * TRAIN_RATIO)
        valid_end = train_end + int(total * VALID_RATIO)

        mapping = {
            "train": all_images[:train_end],
            "valid": all_images[train_end:valid_end],
            "test": all_images[valid_end:],
        }

    total = sum(len(images) for images in mapping.values())
    print(f"Status: total images to prepare = {total:,}")
    print(
        "Status: target split counts -> "
        + ", ".join(f"{split}={len(images):,}" for split, images in mapping.items())
    )

    ensure_empty_split_dirs(work_dir)
    image_stats = {
        "copied_jpeg": 0,
        "copied_jpeg_unverified": 0,
        "converted_to_jpg": 0,
    }

    processed = 0
    with tqdm(
        total=total,
        unit="img",
        dynamic_ncols=True,
        mininterval=0.5,
        desc="Preparing JPG dataset",
    ) as progress_bar:
        for split, images in mapping.items():
            print(f"Status: writing {split} split ({len(images):,} images)")
            for image_path in images:
                dst_image = work_dir / split / "images" / jpg_name_for_image(image_path)
                action = prepare_image_as_jpg(image_path, dst_image)
                image_stats[action] += 1

                src_label = label_for_image(image_path)
                dst_label = work_dir / split / "labels" / f"{image_path.stem}.txt"
                if dst_label.exists():
                    raise FileExistsError(f"Label target already exists: {dst_label}")
                if src_label.exists():
                    link_or_copy(src_label, dst_label)
                else:
                    dst_label.touch()

                processed += 1
                progress_bar.update(1)
                if processed % PROGRESS_REPORT_EVERY == 0 or processed == total:
                    print(f"Status: prepared {processed:,}/{total:,} images")

            print(f"Status: {split} split completed")

    target_dir = work_dir
    final_counts = split_counts(target_dir)
    print("Status: JPG preparation completed")
    print(
        "Status: image normalization summary -> "
        + ", ".join(
            [
                f"copy JPEG={image_stats['copied_jpeg']:,}",
                f"copy JPEG (unverified)={image_stats['copied_jpeg_unverified']:,}",
                f"converted to JPG={image_stats['converted_to_jpg']:,}",
            ]
        )
    )

    prepared_yaml = {
        "path": str(target_dir),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": raw_data["nc"],
        "names": raw_data["names"],
    }

    prepared_yaml_path = target_dir / "data.yaml"
    prepared_yaml_path.write_text(
        yaml.safe_dump(prepared_yaml, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print(f"Status: wrote training yaml -> {prepared_yaml_path}")

    return target_dir, prepared_yaml_path, final_counts, image_stats


prepared_dir, prepared_yaml_path, counts, image_stats = prepare_dataset(raw_dataset_dir, WORK_DIR, raw_data)
total_images = sum(counts.values())
normalization_summary = [
    f"copy JPEG={image_stats['copied_jpeg']:,}",
    f"copy JPEG (unverified)={image_stats['copied_jpeg_unverified']:,}",
    f"converted to JPG={image_stats['converted_to_jpg']:,}",
]

report_lines = [
    "## 1. Dataset Summary",
    f"- Total images: **{total_images:,}**",
    (
        f"- Split counts: Train (**{counts['train']:,} | {counts['train'] / total_images * 100:.1f}%**) "
        f"| Valid (**{counts['valid']:,} | {counts['valid'] / total_images * 100:.1f}%**) "
        f"| Test (**{counts['test']:,} | {counts['test'] / total_images * 100:.1f}%**)"
    ),
    f"- Number of classes: **{raw_data['nc']} classes**",
    f"- JPG normalization: **{' | '.join(normalization_summary)}**",
    f"- data.yaml used for training: `{prepared_yaml_path}`",
]
display(Markdown("\n".join(report_lines)))


## ตั้งค่าการเรียนรู้ (Training Setup)

โน้ต:
- ใช้ `yolo11s.pt` เพื่อบาลานซ์ระหว่างความเร็วและความแม่นยำ
- ถ้า Colab RAM / VRAM ไม่พอ ให้ลด `batch` จาก 16 ลงเป็น 8
- ค่า `hsv_v=0.20` ถูกใช้แทนการปรับความสว่างประมาณ `+-20%`


In [ ]:
import torch
from IPython.display import Markdown, display
from ultralytics import YOLO

missing = [name for name in ("BASE_DIR", "SEED") if name not in globals()]
if missing:
    raise RuntimeError("RUN cell ที่เกี่ยวข้องก่อน ; ตัวแปรที่ยังไม่มี: " + ", ".join(missing))

if "prepared_yaml_path" not in globals():
    candidate_paths = []
    if "WORK_DIR" in globals():
        candidate_paths.append(WORK_DIR / "data.yaml")
    if "RAW_DIR" in globals():
        candidate_paths.append(RAW_DIR / "data.yaml")

    prepared_yaml_path = next((path for path in candidate_paths if path.exists()), None)
    if prepared_yaml_path is None:
        raise RuntimeError("RUN ที่เกี่ยวข้องก่อย เพื่อสร้าง data.yaml สำหรับ train")
    print(f"Status: recovered training yaml -> {prepared_yaml_path}")

DEVICE_MODE = "AUTO"  # เลือก "AUTO", "GPU" หรือ "CPU"
MODEL_NAME = "yolo11s.pt"


def resolve_training_device(mode: str):
    normalized_mode = mode.strip().upper()
    valid_modes = {"AUTO", "GPU", "CPU"}
    if normalized_mode not in valid_modes:
        raise ValueError(f"DEVICE_MODE ต้องเป็นหนึ่งใน {sorted(valid_modes)}")

    cuda_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if cuda_available else None

    if normalized_mode == "GPU":
        if not cuda_available:
            raise RuntimeError("ผู้ใช้เลือก GPU แต่ไม่พบ CUDA GPU ในเครื่องนี้")
        return 0, "GPU", gpu_name

    if normalized_mode == "CPU":
        return "cpu", "CPU", gpu_name

    if cuda_available:
        return 0, "GPU", gpu_name
    return "cpu", "CPU", gpu_name


resolved_device, resolved_device_label, detected_gpu_name = resolve_training_device(DEVICE_MODE)
TRAIN_ARGS = {
    "data": str(prepared_yaml_path),
    "epochs": 120,
    "batch": 20,
    "imgsz": 640,
    "project": str(BASE_DIR / "runs"),
    "name": "yolo11_fruit_detector",
    "exist_ok": True,
    "seed": SEED,
    "device": resolved_device,
    "patience": 20,
    "degrees": 5.0,
    "translate": 0.05,
    "scale": 0.10,
    "fliplr": 0.5,
    "hsv_v": 0.15,
    "hsv_s": 0.20,
    "mosaic": 0.20,
    "mixup": 0.0,
    "plots": True,
}

augmentation_summary = (
    "Moderate augmentation: rotation 5 deg, translation 5%, scale 10%, horizontal flip 50%, "
    "Mosaic 20%, brightness jitter 15% (hsv_v=0.15), and saturation jitter 20% (hsv_s=0.20)"
)

device_summary = (
    f"**{resolved_device_label}**"
    if resolved_device_label == "CPU"
    else f"**{resolved_device_label}** ({detected_gpu_name})"
)

display(Markdown(
    "## 2. การตั้งค่าการเรียนรู้ (Training Setup)\n"
    f"- โหมดอุปกรณ์ที่เลือก: **{DEVICE_MODE}**\n"
    f"- อุปกรณ์ที่ใช้ train: {device_summary}\n"
    f"- CUDA พร้อมใช้งาน: **{torch.cuda.is_available()}**\n"
    f"- Augmentation ที่ใช้: **{augmentation_summary}**\n"
    f"- จำนวน Epochs: **{TRAIN_ARGS['epochs']}** รอบ | Batch Size: **{TRAIN_ARGS['batch']}**\n"
    f"- โมเดลตั้งต้น: **{MODEL_NAME}**\n"
    f"- โฟลเดอร์บันทึกผล: `{TRAIN_ARGS['project']}`"
))

print(f"Status: device mode = {DEVICE_MODE}")
if resolved_device_label == "GPU":
    print(f"Status: using GPU -> {detected_gpu_name} (device={resolved_device})")
else:
    print("Status: using CPU")


In [ ]:
from pathlib import Path

missing = [name for name in ("TRAIN_ARGS", "MODEL_NAME", "detected_gpu_name") if name not in globals()]
if missing:
    raise RuntimeError("กรุณารัน cell 11 ก่อน cell 12; ตัวแปรที่ยังไม่มี: " + ", ".join(missing))

print("Status: starting training...")
print(f"Status: training outputs will be saved to {TRAIN_ARGS['project']}")
if TRAIN_ARGS["device"] == "cpu":
    print("Status: confirmed training device -> CPU")
else:
    print(f"Status: confirmed training device -> GPU ({detected_gpu_name})")
    print(f"Status: CUDA device count = {torch.cuda.device_count()}")

model = YOLO(MODEL_NAME)
train_results = model.train(**TRAIN_ARGS)

run_dir = Path(TRAIN_ARGS["project"]) / TRAIN_ARGS["name"]
best_model_path = run_dir / "weights" / "best.pt"

print("Training เสร็จแล้ว")
print("Best model:", best_model_path)


In [ ]:
from pathlib import Path

import yaml
from IPython.display import Image, display

candidate_run_dirs = []
if "TRAIN_ARGS" in globals():
    candidate_run_dirs.append(Path(TRAIN_ARGS["project"]) / TRAIN_ARGS["name"])
if "best_model_path" in globals():
    candidate_run_dirs.append(Path(best_model_path).parent.parent)
if "BASE_DIR" in globals():
    candidate_run_dirs.append(Path(BASE_DIR) / "runs" / "yolo11_fruit_detector")
candidate_run_dirs.append(Path.cwd() / "runtime_data" / "runs" / "yolo11_fruit_detector")

run_dir = next((path for path in candidate_run_dirs if (path / "args.yaml").exists()), None)
if run_dir is None:
    raise FileNotFoundError("ไม่พบโฟลเดอร์ผล train; ให้รัน cell train มาก่อน หรือเช็ก runtime_data/runs")

args_yaml_path = run_dir / "args.yaml"
if "TRAIN_ARGS" not in globals() and args_yaml_path.exists():
    saved_args = yaml.safe_load(args_yaml_path.read_text(encoding="utf-8")) or {}
    TRAIN_ARGS = {
        "project": str(run_dir.parent),
        "name": run_dir.name,
        "imgsz": int(saved_args.get("imgsz", 640)),
        "batch": int(saved_args.get("batch", 20)),
    }
    print(f"Status: recovered TRAIN_ARGS from -> {args_yaml_path}")

results_png = run_dir / "results.png"
if results_png.exists():
    display(Image(filename=str(results_png)))
else:
    print(f"ยังไม่พบ results.png ที่ {results_png}")


In [ ]:
import gc
from pathlib import Path

import torch
import yaml
from IPython.display import Markdown, display
from ultralytics import YOLO

candidate_run_dirs = []
if "TRAIN_ARGS" in globals():
    candidate_run_dirs.append(Path(TRAIN_ARGS["project"]) / TRAIN_ARGS["name"])
if "best_model_path" in globals():
    candidate_run_dirs.append(Path(best_model_path).parent.parent)
if "BASE_DIR" in globals():
    candidate_run_dirs.append(Path(BASE_DIR) / "runs" / "yolo11_fruit_detector")
candidate_run_dirs.append(Path.cwd() / "runtime_data" / "runs" / "yolo11_fruit_detector")

run_dir = next((path for path in candidate_run_dirs if (path / "args.yaml").exists()), None)
if run_dir is None:
    raise FileNotFoundError("ไม่พบโฟลเดอร์ผล train; ให้รัน cell train มาก่อน หรือเช็ก runtime_data/runs")

args_yaml_path = run_dir / "args.yaml"
saved_args = yaml.safe_load(args_yaml_path.read_text(encoding="utf-8")) if args_yaml_path.exists() else {}
saved_args = saved_args or {}

if "TRAIN_ARGS" not in globals():
    TRAIN_ARGS = {
        "project": str(run_dir.parent),
        "name": run_dir.name,
        "imgsz": int(saved_args.get("imgsz", 640)),
        "batch": int(saved_args.get("batch", 20)),
    }
    print(f"Status: recovered TRAIN_ARGS from -> {args_yaml_path}")

if "prepared_yaml_path" not in globals():
    candidate_yaml_paths = []
    if "prepared_dir" in globals():
        candidate_yaml_paths.append(Path(prepared_dir) / "data.yaml")
    if "WORK_DIR" in globals():
        candidate_yaml_paths.append(Path(WORK_DIR) / "data.yaml")
    if saved_args.get("data"):
        candidate_yaml_paths.append(Path(saved_args["data"]))
    candidate_yaml_paths.append(Path.cwd() / "runtime_data" / "custom_data_yolo11" / "data.yaml")

    prepared_yaml_path = next((path for path in candidate_yaml_paths if path.exists()), None)
    if prepared_yaml_path is None:
        raise FileNotFoundError("ไม่พบ data.yaml ของ prepared dataset; ให้รัน cell เตรียม dataset ก่อนอย่างน้อยหนึ่งครั้ง")
    print(f"Status: recovered prepared_yaml_path -> {prepared_yaml_path}")

if "best_model_path" not in globals():
    candidate_model_paths = []
    if saved_args.get("project") and saved_args.get("name"):
        candidate_model_paths.append(Path(saved_args["project"]) / saved_args["name"] / "weights" / "best.pt")
    candidate_model_paths.append(run_dir / "weights" / "best.pt")

    best_model_path = next((path for path in candidate_model_paths if path.exists()), None)
    if best_model_path is None:
        raise FileNotFoundError("ไม่พบ best.pt; ให้รัน cell train มาก่อน หรือเช็กไฟล์ใน runtime_data/runs")
    print(f"Status: recovered best_model_path -> {best_model_path}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

best_model = YOLO(str(best_model_path))
val_batch = max(1, min(int(TRAIN_ARGS["batch"]), 8))
metrics = best_model.val(
    data=str(prepared_yaml_path),
    split="test",
    imgsz=int(TRAIN_ARGS["imgsz"]),
    batch=val_batch,
    plots=True,
)

map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)
gap = map50 - map50_95

if gap >= 0.20:
    gap_analysis = (
        "ค่า mAP@50-95 ต่ำกว่า mAP@50 มาก แปลว่าโมเดลมักตรวจพบวัตถุได้ แต่ตำแหน่งกรอบยังไม่แม่นพอเมื่อใช้ IoU ที่เข้มขึ้น "
        "สาเหตุที่เป็นไปได้คือวัตถุหลายคลาสมีลักษณะคล้ายกัน, ขอบวัตถุบางภาพไม่ชัด, หรือ annotation ของบางคลาสตีกรอบไม่สม่ำเสมอ"
    )
elif gap >= 0.10:
    gap_analysis = (
        "โมเดลเริ่มแยกคลาสได้ดีในระดับหนึ่ง แต่ความละเอียดของกรอบยังมีช่องว่างอยู่ "
        "ควรลองเพิ่มคุณภาพ annotation, เพิ่มจำนวนภาพต่อคลาสที่คล้ายกัน, หรือเพิ่มภาพที่มีมุมมองหลากหลาย"
    )
else:
    gap_analysis = (
        "ช่องว่างระหว่าง mAP@50 และ mAP@50-95 ไม่มาก แปลว่าทั้งการตรวจพบและการตีกรอบมีความสม่ำเสมอค่อนข้างดี"
    )

display(Markdown(
    "## 3. วิเคราะห์ผลลัพธ์เชิงสถิติ (Performance Analysis)\n"
    f"- Validation batch ที่ใช้: **{val_batch}**\n"
    f"- ค่า mAP@50: **{map50:.4f}**\n"
    f"- ค่า mAP@50-95: **{map50_95:.4f}**\n"
    f"- วิเคราะห์ความต่าง: {gap_analysis}"
))


In [ ]:
import gc
import glob
from pathlib import Path

import torch
from IPython.display import Image, display
from ultralytics import YOLO

if "prepared_dir" not in globals():
    candidate_dirs = []
    if "prepared_yaml_path" in globals():
        candidate_dirs.append(Path(prepared_yaml_path).parent)
    if "WORK_DIR" in globals():
        candidate_dirs.append(Path(WORK_DIR))
    candidate_dirs.append(Path.cwd() / "runtime_data" / "custom_data_yolo11")

    prepared_dir = next((path for path in candidate_dirs if (path / "data.yaml").exists()), None)
    if prepared_dir is None:
        raise FileNotFoundError("ไม่พบ prepared dataset; ให้รัน cell เตรียม dataset ก่อนอย่างน้อยหนึ่งครั้ง")
    print(f"Status: recovered prepared_dir -> {prepared_dir}")

if "best_model" not in globals():
    candidate_models = []
    if "best_model_path" in globals():
        candidate_models.append(Path(best_model_path))
    if "TRAIN_ARGS" in globals():
        candidate_models.append(Path(TRAIN_ARGS["project"]) / TRAIN_ARGS["name"] / "weights" / "best.pt")
    candidate_models.append(Path.cwd() / "runtime_data" / "runs" / "yolo11_fruit_detector" / "weights" / "best.pt")

    best_model_path = next((path for path in candidate_models if path.exists()), None)
    if best_model_path is None:
        raise FileNotFoundError("ไม่พบ best.pt; ให้รัน cell train มาก่อน หรือเช็กไฟล์ใน runtime_data/runs")
    best_model = YOLO(str(best_model_path))
    print(f"Status: loaded best model -> {best_model_path}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

test_images = sorted(glob.glob(str(prepared_dir / "test" / "images" / "*")))
sample_images = test_images[:300] # จำกัดจำนวนภาพตัวอย่างสำหรับการแสดงผล (ปรับได้ตามต้องการ แต่ระวงเรื่อง VRAM ไม่ถูกล้างถึงเมื่อ Run เสร็จแล้ว ให้ปิด IDE ใหม่เพื่อเคลียร์ VRAM ให้หมดก่อนเปิดใหม่)

if not sample_images:
    print("No images found in test split")
else:
    prediction_results = best_model.predict(
        source=sample_images,
        conf=0.25,
        save=True,
        line_width=2,
        batch=1,
    )
    prediction_dir = Path(prediction_results[0].save_dir)

    for image_path in sorted(prediction_dir.glob("*"))[:300]: # จำกัดจำนวนภาพตัวอย่างสำหรับการแสดงผล (ปรับได้ตามต้องการ)
        display(Image(filename=str(image_path), width=500))
